# MERGE JUPITER TOOL 

In [ ]:
import os
import re
import time
from pathlib import Path
import pandas as pd
import fastexcel  # pip install fastexcel polars xlsxwriter tkinterdnd2
import polars as pl
import xlsxwriter
import tkinter as tk
from tkinter import filedialog

In [ ]:
FACTORY_ORDER = ["VH", "VH2", "JV", "JV2"]

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame) and "Factory" in value.columns:
        sorted_df = value.copy()
        sorted_df["Factory"] = pd.Categorical(
            sorted_df["Factory"],
            categories=FACTORY_ORDER,
            ordered=True,
        )
        globals()[name] = (
            sorted_df.sort_values("Factory", kind="stable")
            .reset_index(drop=True)
        )

In [ ]:

try:
    from tkinterdnd2 import TkinterDnD, DND_FILES
    HAS_DND = True
except ImportError:
    HAS_DND = False


# --- Chọn thư mục (kéo-thả / Browse / dán đường dẫn) ---------------------
def pick_folder() -> str:
    result = {"path": ""}

    root = TkinterDnD.Tk() if HAS_DND else tk.Tk()
    root.title("Merge Excel")
    root.geometry("520x240")
    root.attributes("-topmost", True)

    msg = ("KÉO & THẢ thư mục chứa các subfolder vào đây"
           if HAS_DND else
           "(Chưa cài tkinterdnd2 -> dùng Browse hoặc dán đường dẫn)")
    drop_area = tk.Label(root, text=msg, relief="groove", bd=2,
                         bg="#eef3fb", font=("Segoe UI", 11))
    drop_area.pack(fill="both", expand=True, padx=12, pady=(12, 6))

    path_var = tk.StringVar()
    tk.Entry(root, textvariable=path_var).pack(fill="x", padx=12)

    def accept(path: str):
        path = path.strip().strip('"')
        if os.path.isfile(path):          # thả nhầm file -> lấy thư mục cha
            path = os.path.dirname(path)
        if os.path.isdir(path):
            result["path"] = os.path.abspath(path)
            root.quit()
        else:
            drop_area.config(text=f"Đường dẫn không hợp lệ:\n{path}")

    def on_drop(event):
        items = root.tk.splitlist(event.data)  # xử lý đường dẫn có khoảng trắng {..}
        if items:
            accept(items[0])

    def browse():
        p = filedialog.askdirectory(parent=root, title="Chọn thư mục")
        if p:
            accept(p)

    if HAS_DND:
        drop_area.drop_target_register(DND_FILES)
        drop_area.dnd_bind("<<Drop>>", on_drop)

    btns = tk.Frame(root)
    btns.pack(pady=8)
    tk.Button(btns, text="Browse...", width=12, command=browse).pack(side="left", padx=4)
    tk.Button(btns, text="OK", width=12,
              command=lambda: accept(path_var.get())).pack(side="left", padx=4)

    root.protocol("WM_DELETE_WINDOW", root.quit)
    root.mainloop()
    root.withdraw()   # ẩn ngay lập tức
    root.update()     # xử lý nốt sự kiện còn lại
    root.destroy()
    return result["path"]


root_folder = pick_folder()
if not root_folder:
    raise RuntimeError("Bạn chưa chọn thư mục.")

print(f"Thư mục nguồn: {root_folder}")

In [ ]:
# --- Cấu hình -----------------------------------------------------------
output_file = os.path.join(
    os.path.dirname(root_folder), "MERGED_FILE", "Merged_report.xlsx"
)

ROWS_TO_DROP = 3  # số dòng đầu bị xóa trước khi merge (dòng kế tiếp làm header)
EXCEL_ROW_LIMIT = 1_048_576


# --- Thu thập dữ liệu ---------------------------------------------------
frames, matched_files, failed_files, multi_sheet_files = [], [], [], []
output_abspath = os.path.abspath(output_file)
start_time = time.time()

for current_root, dirs, files in os.walk(root_folder):
    dirs[:] = [d for d in dirs if d.casefold() != "merged_file"]

    for filename in files:
        if filename.startswith("~$") or not filename.casefold().endswith((".xlsx", ".xls")):
            continue

        file_path = os.path.join(current_root, filename)
        if os.path.abspath(file_path) == output_abspath:
            continue

        file_start = time.time()

        try:
            xl = fastexcel.read_excel(file_path)
            sheet = xl.sheet_names[0]  # file chỉ có 1 sheet -> đọc sheet đầu tiên

            if len(xl.sheet_names) > 1:
                multi_sheet_files.append((file_path, xl.sheet_names))
                print(f"[CHÚ Ý] File có {len(xl.sheet_names)} sheet, chỉ đọc '{sheet}': {file_path}")

            # header_row=3 -> bỏ 3 dòng đầu, dòng thứ 4 làm tên cột
            df = xl.load_sheet_by_name(sheet, header_row=ROWS_TO_DROP).to_polars()

            matched_files.append(file_path)
            # if not df.is_empty():
            #     c_file = "Source_File" if "Source_File" not in df.columns else "Source_File_"
            #     c_sheet = "Source_Sheet" if "Source_Sheet" not in df.columns else "Source_Sheet_"
            #     df = df.select(
            #         pl.lit(filename).alias(c_file),
            #         pl.lit(sheet).alias(c_sheet),
            #         pl.all(),
            #     )
            #    frames.append(df)
            if not df.is_empty():
                folder_name = os.path.basename(current_root)  # tên subfolder chứa file
                c_file = "Source_File" if "Source_File" not in df.columns else "Source_File_"
                c_sheet = "Source_Sheet" if "Source_Sheet" not in df.columns else "Source_Sheet_"
                df = df.select(
                    pl.lit(folder_name).alias(c_file),
                    pl.lit(sheet).alias(c_sheet),
                    pl.all(),
                )
                frames.append(df)

            print(f"[OK] ({time.time() - file_start:.1f}s): {file_path}")

        except Exception as e:
            failed_files.append((file_path, str(e)))
            print(f"[LỖI] Không đọc được {file_path}: {e}")

print(f"\nTổng thời gian đọc: {time.time() - start_time:.1f}s | engine: calamine")

# --- Gộp và ghi kết quả -------------------------------------------------
if frames:
    try:
        merged = pl.concat(frames, how="diagonal_relaxed")
    except Exception as e:
        print(f"[CẢNH BÁO] Gộp theo kiểu thất bại ({e}); chuyển toàn bộ sang String.")
        frames = [f.cast(pl.String, strict=False) for f in frames]
        merged = pl.concat(frames, how="diagonal")

    write_start = time.time()
    os.makedirs(os.path.dirname(output_abspath), exist_ok=True)
    chunk_size = EXCEL_ROW_LIMIT - 1  # trừ 1 dòng header

    xlsx_options = {
        "strings_to_urls": False,
        "strings_to_formulas": False,
        "strings_to_numbers": False,
        "nan_inf_to_errors": True,
    }

    with xlsxwriter.Workbook(output_file, xlsx_options) as wb:
        for i, start in enumerate(range(0, merged.height, chunk_size), start=1):
            sheet_name = "Merged_Data" if i == 1 else f"Merged_Data_{i}"
            merged.slice(start, chunk_size).write_excel(
                workbook=wb,
                worksheet=sheet_name,
                table_style=None,
                autofit=False,
                header_format={"bold": False},
                dtype_formats={
                    pl.Datetime: "yyyy-mm-dd hh:mm:ss",
                    pl.Date: "yyyy-mm-dd",
                },
            )

    print(f"Thời gian ghi: {time.time() - write_start:.1f}s")
    print(f"Đã đọc thành công {len(matched_files)} file.")
    print(f"Tổng số dòng dữ liệu: {merged.height:,}")
    print(f"Đã lưu kết quả tại: {output_abspath}")
else:
    print("Không tìm thấy dữ liệu phù hợp để gộp.")

# --- Báo cáo ------------------------------------------------------------
if multi_sheet_files:
    print(f"\n{len(multi_sheet_files)} file có nhiều hơn 1 sheet (chỉ đọc sheet đầu):")
    for p, sheets in multi_sheet_files:
        print(f"  - {p} (sheet có: {sheets})")

if failed_files:
    print(f"\n{len(failed_files)} file bị lỗi:")
    for p, err in failed_files:
        print(f"  - {p}: {err}")

In [ ]:
# --- Dọn dẹp tkinter ----------------------------------------------------
try:
    if tk._default_root is not None:
        tk._default_root.destroy()
except Exception:
    pass